# TiA 6 — Diffusion: from noise to data

**Big question:** What must a model know to turn noise into data?

A two-component Gaussian mixture makes the forward marginals and their exact score visible. This lets us inspect reverse diffusion without downloading weights or hiding the mechanism in a framework.

## How to work through this activity

This is a guided investigation rather than a coding tutorial. For each experiment:

1. Read the mathematical claim and identify the quantity being measured.
2. Predict the qualitative result before running the code.
3. Run one cell at a time and inspect both values and plots.
4. Change only the suggested variable; rerun and explain what changed.
5. Answer the **Explain** questions in your own words.

The code contains more comments than production software intentionally. You are not expected to memorise framework syntax. Focus on the relationship between assumptions, measurements and conclusions.

## Notation and prediction

The variance-preserving forward process has a closed-form marginal

$$x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\epsilon,\qquad \epsilon\sim\mathcal N(0,I).$$

Its score $s_t(x)=\nabla_x\log p_t(x)$ points toward increasing log-density. For the SDE $dx=-\tfrac{\beta}{2}x\,dt+\sqrt\beta\,dW_t$, reverse-time dynamics use the score to undo diffusion. Predict how the score field and sample quality change with $t$ and with the number of numerical steps.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch

# Fix every random-number generator so that your plots match the reference run.
# After completing the guided activity, change the seed to test robustness.
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

# The default path is designed for a CPU. Set this to False only after the
# notebook works and you want to run longer variants.
FAST_MODE = True

In [ ]:
centres=np.array([[-2.,0.],[2.,0.]])
def alpha(t): return np.exp(-3*t)
def sample_forward(x0,t):
    a=alpha(t); return np.sqrt(a)*x0+np.sqrt(1-a)*rng.normal(size=x0.shape)
x0=centres[rng.integers(0,2,1000)]+.25*rng.normal(size=(1000,2))
fig,ax=plt.subplots(1,5,figsize=(12,2.5))
for a,t in zip(ax,np.linspace(0,1,5)):
    z=sample_forward(x0,t); a.scatter(*z.T,s=3); a.set_title(f"t={t:.2f}"); a.set_xlim(-4,4);a.set_ylim(-3,3)
plt.show()

## Exact score and reverse-time sampling

For this known mixture, we can calculate $
abla_x\log p_t(x)$ exactly. A learned denoiser estimates equivalent information. Euler–Maruyama then combines the score-driven drift with stochastic noise; the network and the sampler are different objects.

In [ ]:
sigma0=.25
def score(x,t):
    a=alpha(t); means=np.sqrt(a)*centres; var=a*sigma0**2+1-a
    logp=-((x[:,None,:]-means[None,:,:])**2).sum(2)/(2*var)
    w=np.exp(logp-logp.max(1,keepdims=True)); w/=w.sum(1,keepdims=True)
    return (w[:,:,None]*(means[None,:,:]-x[:,None,:])/var).sum(1)
def reverse_sample(steps,n=600):
    x=rng.normal(size=(n,2)); dt=1/steps; snapshots=[x.copy()]
    # Reverse SDE for forward dX=-1.5X dt+sqrt(3)dW, integrated from t=1 to 0.
    for k in range(steps,0,-1):
        t=k/steps; x += (1.5*x+3*score(x,t))*dt+np.sqrt(3*dt)*rng.normal(size=x.shape)
        if k in {steps,3*steps//4,steps//2,steps//4,1}: snapshots.append(x.copy())
    return x,snapshots
fig,ax=plt.subplots(1,3,figsize=(9,3))
quality={}
for a,steps in zip(ax,[10,30,100]):
    z,_=reverse_sample(steps)
    # Distance to the nearest component centre is a simple transparent proxy.
    nearest_distance=np.sqrt(((z[:,None,:]-centres[None,:,:])**2).sum(2)).min(1).mean()
    quality[steps]=nearest_distance
    a.scatter(*z.T,s=3); a.set_title(f"{steps} steps\nmean distance={nearest_distance:.2f}");a.set_xlim(-4,4);a.set_ylim(-3,3)
plt.show()
print("Mean distance to nearest target mode (lower is better):",quality)
assert quality[100] < quality[10]-.15

### Real-image bridge (optional)

Use the same questions—intermediate states, fixed initial noise, step count, scheduler—with a [small pretrained DDPM from Diffusers](https://huggingface.co/docs/diffusers/api/pipelines/ddpm). It requires model-weight downloads and is therefore intentionally outside the default notebook.

### Explain

1. What does the score point toward at early versus late noise times?
2. Why can changing the sampler change outputs while the learned network stays fixed?
3. Describe the compute–quality trade-off in the step-count plot.

**Reading:** [Ho et al., Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2006.11239). For the lecture connection to flows, see [Lipman et al., Flow Matching](https://arxiv.org/abs/2210.02747).

## Expected pattern and limits

Forward samples approach isotropic noise. Reverse samples recover two balanced modes, and finer discretisation places samples closer to the target mixture centres. Here the exact score replaces a trained network so the sampling mechanism is visible; real diffusion models must estimate the score or equivalent noise target.